# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Question: which pages are heading for a search-visibility decline (`trend_direction == "down"`)?
That's a yes/no question with an observed label, so this notebook trains **Logistic Regression**,
then a **Random Forest**, and compares both against my Week-4 rule baseline — same data, same
client-grouped split, same metric (precision@K, plus AUC).

Skills used: `training-honest-models` + `flyrank/flyrank-data` (per `skills/README.md`).

## 1. Method choice and why

**The question shape:** "will this page's impressions decline?" is a yes/no question with an
observed label (`is_declining_label = trend_direction == "down"`) already sitting in the data —
per `training-honest-models`, that shape starts with **Logistic Regression, then Random Forest**:
readable first, stronger second, and I only keep the stronger one if it actually earns its
complexity against the same metric.

**Features:** I use FlyRank's own `MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` list
from `scripts/ml_utils.py` (log-transformed traffic counts + current-state metrics + tier
categoricals). `trend_direction` and `trend_pct` are excluded — they're the label's own source
columns, never features, per the data dictionary's leakage warning. I engineer the same
`log1p()` traffic columns the prep step defines, since raw impressions/clicks are heavy-tailed
(the auditing-signals skill's rule: handle heavy tails before modeling, not after).

**Why not jump straight to Random Forest / Gradient Boosting:** a depth-2 tree or a linear model
you can read teaches more per point of accuracy than an opaque ensemble two points stronger.
I train the Random Forest anyway so the comparison table can show, honestly, whether the added
complexity actually pays for itself here — not assume it will.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42  # fixed everywhere below, so a rerun reproduces this table
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# --- the label (never a feature source beyond this point) ---
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"{len(df):,} rows | base rate = {df['is_declining_label'].mean():.3f}")

# --- log1p the heavy-tailed traffic counts, matching the prep step's own feature list ---
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col])

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

leakage_cols = {"trend_direction", "trend_pct"}
assert leakage_cols.isdisjoint(MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES), \
    "Leakage: a label-source column ended up in the feature list!"
print("Leakage check passed — no trend_direction / trend_pct in the feature list.")


30,000 rows | base rate = 0.542
Leakage check passed — no trend_direction / trend_pct in the feature list.


## 2. Split design

**Grouped by `client_id`**, not a plain random split. Pages from the same client share a domain,
templates, and site-wide factors — a random row split would let one client's pages sit in both
train and test, which leaks client-level signal into "test" performance. The data dictionary is
explicit about this: use `client_id` for grouped train/test splits, never as a feature. I use
`GroupShuffleSplit` (20% of *clients* held out, not 20% of rows) with a fixed seed, and verify
zero client overlap below.

In [2]:
X = df[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES].copy()
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"train rows: {len(train_idx):,} ({groups.iloc[train_idx].nunique()} clients)")
print(f"test rows:  {len(test_idx):,} ({groups.iloc[test_idx].nunique()} clients)")
print(f"client overlap between train and test: {len(overlap)}")
print(f"train base rate: {y_train.mean():.3f}  |  test base rate: {y_test.mean():.3f}")

# honest caveat: a client-grouped split can starve a fold of a whole category by chance
print()
print("content_type coverage — train:")
print(df["content_type"].iloc[train_idx].value_counts())
print("content_type coverage — test:")
print(df["content_type"].iloc[test_idx].value_counts())


train rows: 23,837 (25 clients)
test rows:  6,163 (7 clients)
client overlap between train and test: 0
train base rate: 0.550  |  test base rate: 0.511

content_type coverage — train:
content_type
keyword article       21044
feedly article         2096
comparison article      697
Name: count, dtype: int64
content_type coverage — test:
content_type
keyword article    6163
Name: count, dtype: int64


With this seed, every `feedly article` and `comparison article` row happens to land in
**train**, none in test — the two clients that publish those types weren't drawn into the 20%
test group. That's a real cost of grouping by client on only 32 clients: it protects against
leakage but can leave a fold blind to a whole category by chance. I flag this now so it isn't
mistaken for a model result later — this split can't tell me how the model does on those two
content types, only on `keyword article`. (Noted again in Section 4.)

## 3. Train + compare vs my baseline

Same test rows, same label, same metric — precision@K plus the base rate — for three things:
my Week-4 rule baseline (recomputed here, deterministically, on the test rows only), Logistic
Regression, and Random Forest.

In [3]:
# --- recompute the Week-4 baseline rule (deterministic, no training — same logic as w04) ---
has_position = df["position_tier"] != "no_data"
bench = (
    df[has_position].groupby("position_tier")
      .apply(lambda x: 100 * x["clicks_90d"].sum() / x["impressions_90d"].sum())
)
bench_map = bench.to_dict()

df["visible"] = (df["impressions_90d"] >= 300).astype(int)
df["bench_ctr"] = df["position_tier"].map(bench_map)
df["ctr_gap_ratio"] = np.where(
    has_position & df["bench_ctr"].gt(0),
    np.clip((df["bench_ctr"] - df["ctr"]) / df["bench_ctr"], 0, None),
    np.nan,
)

def assign_reason(row):
    if row["visible"] == 0:
        return "low_visibility"
    if row["position_tier"] == "no_data" or pd.isna(row["ctr_gap_ratio"]):
        return "no_position_data"
    if row["ctr_gap_ratio"] > 0.30:
        return "ctr_gap_high_visibility"
    elif row["ctr_gap_ratio"] > 0:
        return "ctr_gap_moderate"
    return "on_par_or_above"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["baseline_score"] = np.where(
    df["reason_code"].isin(["ctr_gap_high_visibility", "ctr_gap_moderate"]),
    df["ctr_gap_ratio"] * np.log1p(df["impressions_90d"]),
    0.0,
)
baseline_test_scores = df["baseline_score"].iloc[test_idx].values


In [4]:
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), MODEL_NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), MODEL_CATEGORICAL_FEATURES),
])

logreg = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
logreg.fit(X_train, y_train)
p_logreg = logreg.predict_proba(X_test)[:, 1]

rf = Pipeline([("pre", preprocess), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1,
))])
rf.fit(X_train, y_train)
p_rf = rf.predict_proba(X_test)[:, 1]

print("AUC — baseline rule:      ", round(roc_auc_score(y_test, baseline_test_scores), 3))
print("AUC — logistic regression:", round(roc_auc_score(y_test, p_logreg), 3))
print("AUC — random forest:      ", round(roc_auc_score(y_test, p_rf), 3))


AUC — baseline rule:       0.545
AUC — logistic regression: 0.616
AUC — random forest:       0.61


In [5]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

rows = []
for k in [50, 100, 200, 500]:
    rows.append({
        "k": k,
        "base_rate": round(y_test.mean(), 3),
        "baseline_rule": round(precision_at_k(y_test.values, baseline_test_scores, k), 3),
        "logistic_regression": round(precision_at_k(y_test.values, p_logreg, k), 3),
        "random_forest": round(precision_at_k(y_test.values, p_rf, k), 3),
    })

comparison_table = pd.DataFrame(rows)
comparison_table


,k,base_rate,baseline_rule,logistic_regression,random_forest
0,50,0.511,0.640,0.720,0.580
1,100,0.511,0.660,0.700,0.580
2,200,0.511,0.660,0.710,0.580
3,500,0.511,0.604,0.658,0.604


**Reading the table honestly:** logistic regression beats the Week-4 rule baseline at every
K (e.g. precision@50: rule 0.64 vs. logreg 0.72, against a 0.51 base rate) — a real, if modest,
win from adding a learned model. The random forest, despite being the "stronger" method on paper,
**underperforms both the baseline rule and logistic regression** at every K here (precision@50:
0.58). That's the finding, not a bug to explain away: with only 32 grouped clients and a modest
feature set, the extra flexibility of a forest doesn't pay for itself — it's fitting noise the
linear model doesn't. Per `training-honest-models`: don't reward complexity alone. Logistic
regression is the one I'd actually ship.

## 4. Errors and interpretation

*Where is the model most wrong, what does it lean on, and three concrete hard cases —*
*for the logistic regression model, since it's the one that actually beat the baseline.*

In [6]:
perm = permutation_importance(
    logreg, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1
)
importance = (
    pd.Series(perm.importances_mean, index=X_test.columns)
      .sort_values(ascending=False)
      .head(8)
      .round(4)
)
importance


log_impressions_90d    0.0791
log_clicks_90d         0.0571
impression_tier        0.0357
content_age_days       0.0352
avg_position           0.0266
log_sessions_90d       0.0236
days_with_sessions     0.0072
word_count             0.0070
dtype: float64

**What it leans on:** `log_impressions_90d` and `log_clicks_90d` dominate, followed by
`impression_tier`, `content_age_days`, and `avg_position`. All five make sense: pages with more
traffic history give the model more to go on, older pages have had more time to drift, and
position tracks visibility directly. None of this is "suspiciously perfect" — permutation
importance shuffles the label's own source columns out of reach entirely (Section 1's assert),
so there's no shortcut hiding here, just plausible traffic and age signals.

In [7]:
errors = X_test.copy()
errors["content_id"] = df["content_id"].iloc[test_idx].values
errors["y_true"] = y_test.values
errors["p_hat"] = p_logreg
errors["impressions_90d_raw"] = df["impressions_90d"].iloc[test_idx].values
errors["pred_label"] = (errors["p_hat"] >= 0.5).astype(int)
errors["correct"] = (errors["pred_label"] == errors["y_true"]).astype(int)

print("Accuracy by freshness_tier (test set):")
print(errors.groupby("freshness_tier")["correct"].agg(["mean", "size"]).round(3))
print()

false_positives = errors[(errors["y_true"] == 0) & (errors["p_hat"] > 0.7)].sort_values("p_hat", ascending=False)
false_negatives = errors[(errors["y_true"] == 1) & (errors["p_hat"] < 0.3)].sort_values("p_hat")
print(f"confident false positives: {len(false_positives)}  |  confident false negatives: {len(false_negatives)}")
cols = ["content_id", "p_hat", "ctr", "avg_position", "position_tier", "impressions_90d_raw"]
print()
print("3 confident false positives (model said 'declining', it wasn't):")
print(false_positives[cols].head(3).to_string(index=False))
print()
print("3 confident false negatives (model said 'stable', it was declining):")
print(false_negatives[cols].head(3).to_string(index=False))


Accuracy by freshness_tier (test set):
                 mean  size
freshness_tier             
0-30            0.599  4895
31-90           0.640    50
91-180          0.536  1218

confident false positives: 571  |  confident false negatives: 190

3 confident false positives (model said 'declining', it wasn't):
          content_id    p_hat  ctr  avg_position position_tier  impressions_90d_raw
content_7be5f150dc65 0.954355  0.0           5.9        page_1                  290
content_41baf0722ad9 0.942502  0.0          12.8      striking                 3115
content_5d5653c4eb4f 0.932539  0.0           5.7        page_1                15101

3 confident false negatives (model said 'stable', it was declining):
          content_id    p_hat  ctr  avg_position position_tier  impressions_90d_raw
content_d1e915d03c28 0.063416  0.0          45.0      page_3_5                    2
content_e18144cbd19d 0.066244  0.0           2.0         top_3                    3
content_c268b1716236 0.079146 

**Three concrete hard cases, and why they're hard:**

1. **False positives cluster on high-traffic, zero-click pages** (e.g. `page_1` position with
   15,101 impressions and 0.0% CTR). The model reads "lots of impressions, terrible CTR" as a
   decline signal — the same pattern the Week-4 baseline flags as `refresh_now` — but these
   particular pages held flat rather than declined. A CTR problem and an impressions-trend
   problem are related but not the same thing; the model conflates them.
2. **False negatives cluster on already-tiny pages** (2–3 impressions total). With that little
   volume, a page can swing from "up" to "down" on a single-digit change in raw impressions —
   the model has almost no signal to work with at that scale, and neither would a human rule.
3. **Accuracy is lowest in the `91-180` freshness tier** (53.6%, n=1,218) — the same tier that
   Section-1-of-Week-4's audit flagged as having the *highest* raw decline rate. The model
   struggles most exactly where the base rate is most skewed and hardest to separate from noise.

**Split-coverage caveat (from Section 2):** this test split contains zero `feedly article` /
`comparison article` rows, so none of the numbers above say anything about how the model
performs on those two content types — only on `keyword article`. A different seed, or a
stratified-by-content-type variant of the grouped split, would be needed to check that.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` / `client_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.